# Unidad 5 · Bondad de ajuste y normalidad

Cuaderno de la base abierta de ingeniería (Análisis de Datos). Reproduce el ejemplo resuelto (la
pizzería) y la pieza del caso Estampa Sur (tiempos entre pedidos web). Lee los datos de `../datos/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.diagnostic import lilliefors
import matplotlib.pyplot as plt

DATOS = Path("..") / "datos"
RES = {}   # resultados que el verificador compara

## Ejemplo A · ¿Las llamadas por intervalo siguen una Poisson?

Se estima $\lambda$ con la media, se calculan las frecuencias esperadas, se agrupan las clases con
esperada menor que 5 y se pierde un grado de libertad por el parámetro estimado (`ddof=1`).

In [ ]:
ll = pd.read_csv(DATOS / "u05_pizzeria_llamadas.csv")
n = ll.intervalos.sum()
lam = (ll.llamadas * ll.intervalos).sum() / n
p = [stats.poisson.pmf(k, lam) for k in range(4)] + [stats.poisson.sf(3, lam)]   # 0, 1, 2, 3 y «4 o más»
obs = list(ll.intervalos[:4]) + [ll.intervalos[4:].sum()]
esp = n * np.array(p)
prueba = stats.chisquare(obs, esp, ddof=1)       # gl = 5 - 1 - 1 = 3
print(f"lambda = {lam:.2f}; esperadas = {np.round(esp, 2)}")
print(f"chi2 = {prueba.statistic:.3f}; p = {prueba.pvalue:.3f}")
RES.update(ej_lambda=lam, ej_chi2=prueba.statistic, ej_p=prueba.pvalue)

## Ejemplo B · ¿Los tiempos entre llamadas son exponenciales de media 9 minutos?

La media de 9 minutos viene del historial de la central, no de estos datos: la distribución queda
completamente especificada y el KS clásico vale.

In [ ]:
t = pd.read_csv(DATOS / "u05_pizzeria_tiempos.csv").minutos
ks = stats.kstest(t, "expon", args=(0, 9))
sh = stats.shapiro(t)
print(f"KS: D = {ks.statistic:.4f}; p = {ks.pvalue:.3f}")
print(f"Shapiro-Wilk: W = {sh.statistic:.3f}; p = {sh.pvalue:.3f}")
RES.update(ej_ks_D=ks.statistic, ej_ks_p=ks.pvalue, ej_sh_W=sh.statistic, ej_sh_p=sh.pvalue)

## Caso · Estampa Sur, tiempos entre pedidos web

KS contra la exponencial con la media de los datos (clásico y Lilliefors), chi-cuadrado por clases,
Anderson-Darling, Shapiro-Wilk y los dos gráficos Q-Q.

In [ ]:
x = pd.read_csv(DATOS / "llegadas.csv").minutos
media = x.mean()
ks = stats.kstest(x, "expon", args=(0, media))      # p optimista: la media salió de los mismos datos
lf = lilliefors(x, dist="exp")                      # corrige por el parámetro estimado
bordes = [0, 10, 20, 30, 45, 60, np.inf]
obs = np.histogram(x, bins=bordes)[0]
esp = len(x) * np.diff(stats.expon.cdf(bordes, scale=media))
chi = stats.chisquare(obs, esp, ddof=1)             # 6 clases, 1 parámetro: gl = 4
ad = stats.anderson(x, dist="expon", method="interpolate")
sh = stats.shapiro(x)
print(f"media = {media:.2f} min")
print(f"KS clásico: D = {ks.statistic:.4f}, p = {ks.pvalue:.2f} | Lilliefors: p = {lf[1]:.2f}")
print(f"chi2 = {chi.statistic:.2f}, p = {chi.pvalue:.2f} | observadas {obs} | esperadas {np.round(esp, 2)}")
print(f"Anderson-Darling: A2 = {ad.statistic:.3f}, p >= {ad.pvalue:.2f}")
print(f"Shapiro-Wilk: W = {sh.statistic:.3f}, p = {sh.pvalue:.1e}")
RES.update(caso_ks_D=ks.statistic, caso_ks_p=ks.pvalue, caso_lf_p=lf[1], caso_chi2=chi.statistic,
           caso_chi_p=chi.pvalue, caso_ad=ad.statistic, caso_sh_W=sh.statistic, caso_sh_p=sh.pvalue)

In [ ]:
pos = (np.arange(1, len(x) + 1) - 0.5) / len(x)
xs = np.sort(x)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].plot(stats.expon.ppf(pos, scale=media), xs, "o"); ax[0].axline((0, 0), slope=1, color="C1")
ax[0].set_title("Q-Q contra la exponencial")
ax[1].plot(stats.norm.ppf(pos, media, x.std(ddof=1)), xs, "o"); ax[1].axline((0, 0), slope=1, color="C1")
ax[1].set_title("Q-Q contra la normal")
plt.show()